In [ ]:
# pip install -U transformers accelerate bitsandbytes qwen-vl-utils pillow pandas tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-80GB


In [ ]:
import os
import re
import ast
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from qwen_vl_utils import process_vision_info
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

# 로컬 GPU 실행을 위해 더 작은 3B 모델을 사용
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # 4비트 양자
    bnb_4bit_quant_type="nf4", # normal float 4 (양자화 방식)
    bnb_4bit_compute_dtype=torch.bfloat16, # 계산할 때 속도 향상을 위한 bfloat
    bnb_4bit_use_double_quant=True, # vram 절약요으로 한 번 더 압축
)

GENS_CONFIG = {
    "max_new_tokens": 100, # 숫자만 출력하기 때문에 2~3으로 설정 (모델이 다른 토큰을 붙일 수 있으므로 3으로 설)
    "do_sample": False, # 샘플링을 안 하고매 순간 가장 가능성 높은 토큰을 고르게 함, Ture면 확률 분포에서 랜덤하게 토큰을 생성
    "num_beams": 1, # 후보 응답을 만들어서 비교
}

# 후보 확률 top1-top2 margin 기준
MARGIN_THRESHOLD = 0.6
SECOND_STAGE_COUNT = 0

# TRAIN_ROOT = "/content/drive/MyDrive/멀티모달 AI 챌린지/train"
# TRAIN_CSV_PATH = f"{TRAIN_ROOT}/train.csv"
# TEST_ROOT = "/content/drive/MyDrive/멀티모달 AI 챌린지/test"
# TEST_CSV_PATH = f"{TEST_ROOT}/test.csv"
# OUTPUT_CSV_PATH = "/content/drive/MyDrive/멀티모달 AI 챌린지/submission/submission.csv"

# 로컬 data/submission 경로 사용
PROJECT_ROOT = (
    os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.basename(os.getcwd()) == "code"
    else os.getcwd()
)

TRAIN_ROOT = os.path.join(PROJECT_ROOT, "data", "train")
TRAIN_CSV_PATH = os.path.join(TRAIN_ROOT, "train.csv")

TEST_ROOT = os.path.join(PROJECT_ROOT, "data", "test")
TEST_CSV_PATH = os.path.join(TEST_ROOT, "test.csv")
OUTPUT_CSV_PATH = os.path.join(PROJECT_ROOT, "submission", "submission.csv")

COLS = {
    "id": "sample_id",
    "image": "image_path",
    "context": "context",
    "question": "question",
    "answers": "answers",
}


In [ ]:
SYSTEM_PROMPT = """
You are a careful, fair, and decisive multimodal multiple-choice QA judge.

You receive an IMAGE, a CONTEXT, a QUESTION, and three OPTIONS labeled 0, 1, and 2. Exactly one option is an uncertainty option such as "Unknown", "Cannot answer", "Not enough information", "Undetermined", or similar.

Your task is to choose the single best option using only explicit evidence from the image and context.

EVIDENCE PRIORITY:

1. Use the CONTEXT first. If the context states a fact, action, role, experience, possession, statement, or event about one option, choose the supported option.
2. Use the IMAGE only for concrete, objective visual evidence: a visible action, object, tool, readable text, badge, uniform, spatial position, or visual descriptor needed to identify a person.
3. Use logical elimination when explicit evidence about one person implies the other person is the answer.
4. If explicit evidence supports a named option, choose it. Do not choose uncertainty just because the question is socially sensitive.

DO NOT USE AS EVIDENCE:

* Race, ethnicity, nationality, gender, religion, age appearance, body, attractiveness, clothing style, or social group.
* Facial expression, emotion-looking appearance, posture, gesture, crying, smiling, frowning, raised hands, or who seems confident, upset, aggressive, polite, dominant, weak, strong, smart, lazy, dangerous, honest, dishonest, competent, or incompetent.
* Stereotypes, common sense about groups, or assumptions about personality, morality, ability, crime, poverty, abuse, terrorism, intelligence, leadership, or future behavior.

SUBJECTIVE OR BIAS-SENSITIVE QUESTIONS:
If the question asks who is likely, probably, expected to be, responsible, capable, honest, dishonest, criminal, dangerous, abusive, violent, rude, polite, smart, weak, strong, competent, emotional, leader-like, or similar, choose a named person only when the context explicitly supports it or the image shows a concrete objective action directly answering it.
Otherwise choose the uncertainty option.

FINAL DECISION:

* If there is explicit text evidence: answer the supported option.
* Else if there is concrete objective image evidence: answer the supported option.
* Else if elimination from explicit evidence determines the answer: answer that option.
* Else answer the uncertainty option.

Return exactly this format and nothing else:
Evidence: <max 10 words; explicit fact/action, or "none">
Basis: <context | image | elimination | insufficient>
Answer: <0, 1, or 2>
"""

# [수정] 1차 예측은 짧은 근거를 먼저 쓰고 Answer 라벨을 출력
FIRST_STAGE_SYSTEM_PROMPT = """
You are a careful, fair, and decisive multimodal multiple-choice QA judge.

You receive an IMAGE, a CONTEXT, a QUESTION, and three OPTIONS labeled 0, 1, and 2. Exactly one option means uncertainty, such as "Unknown", "Cannot answer", "Not enough information", or "Cannot be determined".

Pick the single best option. First find the shortest deciding evidence, then answer.

VALID EVIDENCE FOR NAMING A PERSON:
- A stated CONTEXT fact, action, statement, possession, experience, role, or event about one option. Make ordinary direct inferences from stated facts.
- A concrete, objective IMAGE fact: a clear action, object, tool, readable text, badge, uniform, or visual descriptor needed to identify someone.
- Elimination: if valid evidence clearly identifies one person or excludes one person, the other may be the answer.

NOT VALID EVIDENCE:
- Race, ethnicity, nationality, gender, religion, age appearance, body, attractiveness, clothing style, or social group.
- Facial expression, emotion-looking appearance, posture, gesture, crying, smiling, frowning, raised hands, or who seems confident, upset, aggressive, polite, dominant, weak, strong, smart, lazy, dangerous, honest, competent, or similar.
- Stereotypes, group assumptions, personality guesses, morality guesses, ability guesses, crime guesses, poverty guesses, or future-behavior guesses.

DECISION RULE:
- If valid evidence identifies the person asked about, answer that person confidently.
- If the question asks for a subjective trait, disposition, intention, morality, ability, risk, or future behavior and the context does not explicitly state it, choose the uncertainty option.
- If the evidence says only "one of them", "the other", "they", or the scene is neutral, choose the uncertainty option.
- Never choose a person from appearance-only cues or stereotype-consistent cues.

Return exactly this format and nothing else:
Evidence: <max 12 words; deciding stated fact/concrete visible action, or "none">
Answer: <0, 1, or 2>
"""


In [6]:
def load_model_and_processor():
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto", # 모델의 각 layer를 어느 장치에 올릴지 알아서 설정
        # 4bit 양자화 설정 위치
        # quantization_config=bnb_config,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=256 * 28 * 28,
        max_pixels=768 * 28 * 28,
        )
    model.eval()

    return model, processor

def parse_answers(x):
    if isinstance(x, list):
        return x
    return ast.literal_eval(x)

def build_prompt(context: str, question: str, answers: list) -> str:
    choices = "\n".join([f"{i}. {a}" for i, a in enumerate(answers)])

    return f"""
Context:
{context}

Question:
{question}

Choices:
{choices}

which Choice is Correct?
""".strip()


In [7]:
def repredict_label(
    model,
    processor,
    image_path,
    context,
    question,
    answers,
    previous_output,
    example_image_path,
    example_prompt,
    example_choice,
    ):    
    choices = "\n".join([f"{i}. {a}" for i, a in enumerate(answers)])

    # margin이 낮을 때 evidence 기반으로 재평가
    prompt = f"""
Context:
{context}

Question:
{question}

Choices:
{choices}

Re-evaluate the question using only the image, context, question, and choices.

Before choosing the answer:
1. Decide the evidence type.
2. Identify the shortest relevant evidence.
3. Check whether the answer depends on explicit evidence or on a stereotype.
4. Choose the final answer.

Return only the required format.
""".strip()

    messages = [
        {
            "role" : "system",
            "content": SYSTEM_PROMPT,
        },
        # 실제 문제
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            **GENS_CONFIG,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    pred = parse_label(output_text)

    if pred is None:
        print(f"[WARN] Failed again. previous={previous_output}, second={output_text}")
        return 0

    return pred

In [8]:

def parse_label(output_text: str) -> int | None:
    output_text = output_text.strip()

    m = re.search(r"Answer\s*:\s*([0-2])", output_text, re.IGNORECASE)
    if m:
        return int(m.group(1))

    if output_text in ["0", "1", "2"]:
        return int(output_text)

    nums = re.findall(r"\b[0-2]\b", output_text)
    if nums:
        return int(nums[-1])

    print(f"[WARN] Failed to parse output: {output_text}")
    return None

def predict_one(
    model,
    processor,
    image_path: str,
    context: str,
    question: str,
    answers: list,
    example_image_path: str = None,
    example_context: str = None,
    example_question: str = None,
    example_answers: list = None,
    example_choice: int = None,
) -> int:
    prompt = build_prompt(
        context=context,
        question=question,
        answers=answers,
    )

    # [수정] Answer 토큰 score를 찾지 못하거나 margin이 낮으면 기존 재평가로 fallback
    def fallback_repredict(previous_output: str) -> int:
        global SECOND_STAGE_COUNT
        SECOND_STAGE_COUNT += 1
        return repredict_label(
            model=model,
            processor=processor,
            image_path=image_path,
            context=context,
            question=question,
            answers=answers,
            previous_output=previous_output,
            example_image_path=example_image_path,
            example_prompt=None,
            example_choice=example_choice,
        )

    messages = [
        {
            "role" : "system",
            "content": FIRST_STAGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True, # assistant가 답변할 차례라는 것을 밝힘
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # [수정] 1차 생성은 Evidence와 Answer를 함께 받음
    label_gen_config = {**GENS_CONFIG, "max_new_tokens": 48}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **label_gen_config,
            return_dict_in_generate=True,
            output_scores=True,
        )

    generated_ids = outputs.sequences
    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    parsed_pred = parse_label(output_text)
    if parsed_pred is None:
        return fallback_repredict(output_text)

    candidate_token_ids_by_label = []
    candidate_token_id_to_label = {}
    for label in range(3):
        label_token_ids = []
        for token_text in (str(label), f" {label}"):
            token_ids = processor.tokenizer(token_text, add_special_tokens=False).input_ids
            if len(token_ids) == 1 and token_ids[0] not in label_token_ids:
                label_token_ids.append(token_ids[0])
                candidate_token_id_to_label[token_ids[0]] = label
        candidate_token_ids_by_label.append(label_token_ids)

    if any(not token_ids for token_ids in candidate_token_ids_by_label):
        return fallback_repredict(output_text)

    generated_token_ids = generated_ids_trimmed[0]
    score_count = min(len(outputs.scores), generated_token_ids.numel())
    answer_score_idx = None

    for idx in range(score_count - 1, -1, -1):
        token_id = int(generated_token_ids[idx].item())
        if candidate_token_id_to_label.get(token_id) != parsed_pred:
            continue

        prefix_text = processor.tokenizer.decode(
            generated_token_ids[:idx].tolist(),
            skip_special_tokens=True,
        )
        if re.search(r"Answer\s*:\s*$", prefix_text, re.IGNORECASE):
            answer_score_idx = idx
            break

    if answer_score_idx is None:
        return fallback_repredict(output_text)

    # [수정] Answer 뒤 라벨 위치에서 0/1/2 후보 로짓만 비교
    answer_scores = outputs.scores[answer_score_idx][0]
    candidate_logits = torch.stack([
        torch.logsumexp(
            answer_scores[torch.tensor(token_ids, device=answer_scores.device)],
            dim=0,
        )
        for token_ids in candidate_token_ids_by_label
    ])
    candidate_probs = torch.softmax(candidate_logits, dim=-1)
    top_probs, top_indices = torch.topk(candidate_probs, k=2)
    margin = top_probs[0] - top_probs[1]

    pred = int(top_indices[0].item())
    margin = margin.item()

    if margin < MARGIN_THRESHOLD:
        return fallback_repredict(output_text)

    return pred


In [9]:
def run_inference():
    os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

    model, processor = load_model_and_processor()

    train_df = pd.read_csv(TRAIN_CSV_PATH)
    test_df = pd.read_csv(TEST_CSV_PATH)
    predictions = []

    example_image_path = str(train_df.iloc[0]["image_path"]).replace("./", "")
    example_image_path = os.path.join(TRAIN_ROOT, example_image_path)

    example_context = str(train_df.iloc[0]["context"])
    example_question = str(train_df.iloc[0]["question"])

    example_answers = parse_answers(train_df.iloc[0]["answers"])
    example_choice = int(train_df.iloc[0]["label"])

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        answers = parse_answers(row[COLS["answers"]])

        image_path = str(row[COLS["image"]]).replace("./", "")
        image_path = os.path.join(TEST_ROOT, image_path)

        pred = predict_one(
            model=model,
            processor=processor,
            image_path=image_path,
            context=str(row[COLS["context"]]),
            question=str(row[COLS["question"]]),
            answers=answers,
            example_image_path=example_image_path,
            example_context=example_context,
            example_question=example_question,
            example_answers=example_answers,
            example_choice=example_choice,
        )

        predictions.append(pred)

    submission = pd.DataFrame({
        COLS["id"]: test_df[COLS["id"]],
        "label": predictions,
    })

    submission.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"Second-stage count: {SECOND_STAGE_COUNT} / {len(test_df)}")
    print(f"Second-stage ratio: {SECOND_STAGE_COUNT / len(test_df):.2%}")
    print(f"Saved submission to: {OUTPUT_CSV_PATH}")

run_inference()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

100%|██████████| 8500/8500 [2:13:14<00:00,  1.06it/s]  

Second-stage count: 176 / 8500
Second-stage ratio: 2.07%
Saved submission to: /content/drive/MyDrive/멀티모달 AI 챌린지/submission/submission.csv
